# Edges, Contours, and Shape Measurement

> **Beginner · Classical measurement**


## Why this matters

Edges reveal intensity change; contours turn a clean binary result into geometric measurements and simple decisions.

**Where it appears:** Quality inspection, counting objects, coin/part measurement, document boundaries, and rule-based shape classification.


## Learning Objectives

- Understand gradient-based edge detection (Sobel) as the foundation of Canny
- Tune Canny's two thresholds deliberately using the gradient histogram, not guessing
- Compare edge detectors on the same image under noise
- Find and filter contours by area, and understand contour hierarchy
- Compute shape descriptors (perimeter, approx-poly vertex count, circularity)
- Classify simple shapes (triangle/rectangle/circle) from contour features alone


## Prerequisites

09 Thresholding and Morphology

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.Sobel`, `cv2.Canny`, `cv2.findContours`, `cv2.contourArea`, `cv2.arcLength`, `cv2.approxPolyDP`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Edge Detection

Edges are places of rapid intensity change. **Sobel** operators approximate
the image gradient in x and y; their magnitude highlights edges but produces
thick, noisy responses. **Canny** edge detection improves on this with
non-maximum suppression (thin edges to 1px) and hysteresis thresholding
(two thresholds: strong edges are kept, weak edges are kept only if
connected to a strong edge) -- which is why Canny needs care in
choosing its low/high thresholds, not random guessing.


### Contours and Shape Analysis

`cv2.findContours` traces boundaries of connected white regions in a binary
image. Raw contours are just point lists -- the real value comes from
derived shape descriptors: area, perimeter, polygon approximation (vertex
count discriminates triangles/rectangles/pentagons), and circularity
(`4*pi*area/perimeter^2`, which approaches 1.0 for a perfect circle). This
notebook builds an actual shape classifier from these descriptors, not
just a contour-drawing demo.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Edge Detection


### 1. Sobel gradients: the building block

Compute x and y gradients separately, then combine into a magnitude image -- this is conceptually what Canny does internally before non-max suppression.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def sobel_magnitude(gray: np.ndarray, ksize: int = 3) -> np.ndarray:
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=ksize)
    magnitude = np.sqrt(gx**2 + gy**2)
    return np.clip(magnitude / magnitude.max() * 255, 0, 255).astype(np.uint8)


gray = cv2.cvtColor(load_real_image("images/objects", "sudoku.png"), cv2.COLOR_BGR2GRAY)
sobel_mag = sobel_magnitude(gray)
show_grid([("grayscale", gray), ("Sobel gradient magnitude", sobel_mag)])

### 2. Choosing Canny thresholds from the data

Rather than guessing Canny's low/high thresholds, derive them from the median gradient intensity -- a well-known, principled heuristic ('auto Canny').


In [ ]:
def auto_canny(gray: np.ndarray, sigma: float = 0.33) -> np.ndarray:
    """Derive Canny thresholds from the image's median intensity instead of guessing."""
    median = np.median(gray)
    lower = int(max(0, (1.0 - sigma) * median))
    upper = int(min(255, (1.0 + sigma) * median))
    return cv2.Canny(gray, lower, upper)


guessed = cv2.Canny(gray, 50, 150)  # arbitrary guess, common in poor-quality tutorials
principled = auto_canny(gray)

show_grid(
    [
        ("guessed thresholds (50,150)", guessed),
        ("median-derived thresholds", principled),
    ]
)

### 3. Robustness to noise: Sobel vs Canny

Canny's Gaussian pre-smoothing and hysteresis thresholding make it noticeably more robust to noise than raw Sobel magnitude thresholding -- demonstrate with a quantitative edge-pixel count comparison.


In [ ]:
noise = np.random.normal(0, 15, gray.shape)
noisy_gray = np.clip(gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)

sobel_on_noisy = sobel_magnitude(noisy_gray)
_, sobel_binary = cv2.threshold(sobel_on_noisy, 60, 255, cv2.THRESH_BINARY)
canny_on_noisy = auto_canny(noisy_gray)

print(f"Sobel-derived binary edge pixels: {int((sobel_binary > 0).sum())}")
print(f"Canny edge pixels:                {int((canny_on_noisy > 0).sum())}")
show_grid(
    [
        ("noisy input", noisy_gray),
        ("thresholded Sobel (noisy)", sobel_binary),
        ("Canny (noisy)", canny_on_noisy),
    ]
)

## Part 2: Contours and Shape Analysis


### 1. Finding and filtering contours

Small noise contours are common after thresholding real images -- always filter by area before doing anything else with detected contours.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def find_significant_contours(binary: np.ndarray, min_area: int = 10000) -> list:
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c for c in contours if cv2.contourArea(c) >= min_area]


scene = load_real_image("images/objects", "geometric_shapes.png")
gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

contours = find_significant_contours(binary)
print(f"Found {len(contours)} significant contours")

overlay = scene.copy()
cv2.drawContours(overlay, contours, -1, (0, 0, 255), 2)
show_grid([("binary mask", binary), ("contours drawn", overlay)])

### 2. Shape descriptors

Compute perimeter, polygon-approximation vertex count, and circularity for each contour -- the features a simple classifier will use next.


In [ ]:
def shape_descriptors(contour) -> dict:
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, closed=True)
    approx = cv2.approxPolyDP(contour, epsilon=0.04 * perimeter, closed=True)
    circularity = (4 * np.pi * area / (perimeter**2)) if perimeter > 0 else 0
    return {
        "area": area,
        "perimeter": perimeter,
        "vertices": len(approx),
        "circularity": circularity,
    }


for i, c in enumerate(contours):
    print(f"contour {i}: {shape_descriptors(c)}")

### 3. A rule-based shape classifier

Combine vertex count and circularity into simple, explainable classification rules -- triangle/rectangle by vertex count, circle by high circularity.


In [ ]:
def classify_shape(contour) -> str:
    d = shape_descriptors(contour)
    # Real-world 3D objects with shadows require adapted rules!
    if d["circularity"] > 0.7:
        return "circle/sphere"
    if d["vertices"] == 3:
        return "triangle"
    if d["vertices"] in [4, 5, 6]:
        return "rectangle/cube"
    return f"polygon({d['vertices']} sides)"


labeled = scene.copy()
for c in contours:
    name = classify_shape(c)
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(labeled, (x, y), (x + w, y + h), (0, 0, 255), 2)
    cv2.putText(
        labeled,
        name,
        (x, max(y - 8, 12)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 0, 255),
        1,
        cv2.LINE_AA,
    )

show_grid([("classified shapes", labeled)], cols=1)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Edge Detection: Edge Pyramiding (Multi-Scale Edge Detection)

Fine details can produce noisy, cluttered edge maps in full resolution. To extract structural features, we build a multi-scale edge detector that computes gradients across downsampled layers and combines them.


In [ ]:
img = load_real_image("images/objects", "sudoku.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Layer 1: High-resolution edges
edges_l1 = cv2.Canny(gray, 50, 150)

# Layer 2: Medium-resolution edges (downsample first, compute edges, then upsample)
down_l2 = cv2.pyrDown(gray)
edges_l2_raw = cv2.Canny(down_l2, 30, 90)
edges_l2 = cv2.pyrUp(edges_l2_raw)

# Combine multi-scale edge maps
combined = cv2.bitwise_or(edges_l1, edges_l2[: gray.shape[0], : gray.shape[1]])

print("Multi-Scale Edge Pyramiding completed.")
show_grid(
    [
        ("Original Grayscale", gray),
        ("Fine Scale (L1)", edges_l1),
        ("Scale Combined Edges", combined),
    ]
)

### Mini Project — Contours and Shape Analysis: Geometric Circularity Metric for Shape Classification

Identifying objects by shape class (e.g., separating circular washers from hexagonal bolts) is a standard inspection task. Here, we compute a contour's circularity metric: $C = \frac{4\pi \times \text{Area}}{\text{Perimeter}^2}$. A perfect circle has a circularity of 1.0, while elongated shapes have values near 0.


In [ ]:
# Generate scene with circle and rectangle shapes
img = np.zeros((300, 300), dtype=np.uint8)
cv2.circle(img, (80, 150), 40, 255, -1)  # Circle
cv2.rectangle(img, (160, 110), (260, 190), 255, -1)  # Rectangle

contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
canvas = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

for idx, c in enumerate(contours):
    area = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)

    if perimeter > 0:
        circularity = (4 * np.pi * area) / (perimeter**2)
        # Classify based on circularity metric threshold
        shape_type = "Circle" if circularity > 0.85 else "Polygon"

        # Calculate centroid to place label text
        M = cv2.moments(c)
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        cv2.putText(
            canvas,
            f"{shape_type} (C={circularity:.2f})",
            (cx - 50, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (0, 0, 255),
            1,
        )

print("Shape metric analysis complete.")
show(canvas, "Shape Classification Overlay")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Edge Detection
1. Implement `sobel_direction(gray)` returning gradient angle in degrees using `cv2.phase`.
2. Sweep `auto_canny`'s `sigma` parameter from 0.1 to 0.6 and describe the trade-off.
3. Add a Gaussian pre-blur step before `auto_canny` and measure how much it reduces noisy edge pixels.

Use the empty cell below to work through them.


#### Solutions — Edge Detection

In [ ]:
# Solution 1: sobel_direction using cv2.phase
def sobel_direction(gray: np.ndarray) -> np.ndarray:
    """Compute gradient orientation angle in degrees."""
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    # Calculate magnitude and angle (in radians by default, or degrees if parameter set)
    magnitude, angle = cv2.cartToPolar(sobelx, sobely, angleInDegrees=True)
    return angle

In [ ]:
# Solution 2: Sweep auto_canny's sigma parameter
# Explanation: The parameter `sigma` controls the Canny threshold margins relative to the median
# intensity. A small `sigma` (e.g. 0.1) tightens the thresholds around the median, generating a highly
# sensitive edge detector with excessive noise. A high `sigma` (e.g. 0.6) widens thresholds,
# resulting in missing weak edges and fragmented boundary segments.

In [ ]:
# Solution 3: Add Gaussian pre-blur step to reduce noise
def auto_canny_denoised(gray: np.ndarray, sigma: float = 0.33) -> np.ndarray:
    """Apply Canny edge detection with pre-filtering noise reduction."""
    # Denoise using Gaussian blur first
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Compute threshold bounds based on median intensity
    v = np.median(blurred)
    lower = int(max(0, (1.0 - sigma) * v))
    upper = int(min(255, (1.0 + sigma) * v))

    return cv2.Canny(blurred, lower, upper)


# Test run
gray_scene = cv2.cvtColor(
    load_real_image("images/objects", "sudoku.png"), cv2.COLOR_BGR2GRAY
)
angles = sobel_direction(gray_scene)
edges = auto_canny_denoised(gray_scene)
print("Sobel angles shape matches input:", angles.shape == gray_scene.shape)

### Exercises — Contours and Shape Analysis
1. Add 'square' as a distinct class from 'rectangle' using the bounding box aspect ratio.
2. Use `cv2.RETR_TREE` instead of `cv2.RETR_EXTERNAL` on an image with a shape containing a hole, and inspect the hierarchy array.
3. Compute and print the centroid of each contour using image moments (`cv2.moments`).

Use the empty cell below to work through them.


#### Solutions — Contours and Shape Analysis

In [ ]:
# Solution 1: Square vs rectangle aspect ratio classification
def classify_quadrilateral(contour: np.ndarray) -> str:
    """Distinguish between a square and a rectangle using aspect ratio."""
    x, y, w, h = cv2.boundingRect(contour)
    aspect_ratio = float(w) / h
    # A square has aspect ratio very close to 1.0 (allow small deviation tolerance)
    if 0.9 <= aspect_ratio <= 1.1:
        return "Square"
    return "Rectangle"


# Solution 2: RETR_TREE hierarchy validation
def analyze_hierarchy() -> None:
    """Analyze contour hierarchy on concentric rings shape."""
    img = np.zeros((200, 200), dtype=np.uint8)
    cv2.circle(img, (100, 100), 60, 255, -1)  # Outer circle
    cv2.circle(img, (100, 100), 30, 0, -1)  # Hole boundary
    cv2.circle(img, (100, 100), 10, 255, -1)  # Inner solid core

    contours, hierarchy = cv2.findContours(img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    print("Total contours found with RETR_TREE:", len(contours))
    print("Hierarchy matrix shape:", hierarchy.shape)
    # The hierarchy matrix contains structural indices: [Next, Previous, First_Child, Parent]


# Solution 3: Centroid of each contour using image moments
def print_contour_centroids(binary_image: np.ndarray) -> None:
    """Locate and print coordinates of centroids for all detected contours."""
    contours, _ = cv2.findContours(
        binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    for i, c in enumerate(contours):
        M = cv2.moments(c)
        if M["m00"] > 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            print(f"Contour {i}: Centroid = ({cx}, {cy})")


# Test run
img = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(img, (10, 10), (90, 50), 255, -1)
contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if contours:
    print("Classified shape:", classify_quadrilateral(contours[0]))
analyze_hierarchy()

## Summary

You can produce useful edges, extract contours, and measure or classify simple shapes while understanding contour hierarchy.

- **Best Practices:** Smooth before differentiating, tune Canny from observed gradients, filter contours by meaningful scale, and keep the original image for drawing.
- **Common Pitfalls:** Using Canny on noisy images, interpreting every contour as an object, and forgetting that contour retrieval mode changes the result.